<a href="https://colab.research.google.com/github/AnuragSivam/localRepo/blob/main/CnbcArticleSQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import random

BASE_URL = "https://www.cnbc.com"

SECTIONS = [
    "world",
    "business",
    "markets",
    "technology",
    "politics",
    "economy"
]

SITEMAP_INDEX = "https://www.cnbc.com/sitemapAll.xml"

MAX_PAGES = 20
DELAY = (1, 3)

HEADERS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)",
    "Mozilla/5.0 (X11; Linux x86_64)"
]

def get_headers():
    return {"User-Agent": random.choice(HEADERS)}

def request(url):
    try:
        return requests.get(url, headers=get_headers(), timeout=10)
    except:
        return None

def sleep():
    time.sleep(random.uniform(*DELAY))

def scrape_sections():
    links = set()

    for section in SECTIONS:
        print(f"\n[SECTION] {section}")

        for page in range(1, MAX_PAGES + 1):
            url = f"{BASE_URL}/{section}/?page={page}"
            res = request(url)

            if not res or res.status_code != 200:
                print("Failed:", url)
                continue

            soup = BeautifulSoup(res.text, "html.parser")

            for a in soup.find_all("a", href=True):
                href = a["href"]

                if href.startswith("https://www.cnbc.com") and href.endswith(".html"):
                    links.add(href.split("?")[0])

            print(f"Page {page} → {len(links)} links")
            sleep()

    return links

def parse_sitemap(url):
    res = request(url)
    if not res:
        return []

    soup = BeautifulSoup(res.text, "xml")

    sitemaps = soup.find_all("sitemap")
    if sitemaps:
        return [tag.loc.text.strip() for tag in sitemaps]

    urls = soup.find_all("url")
    return [tag.loc.text.strip() for tag in urls]

def scrape_sitemaps():
    all_links = set()

    print("\n[SITEMAP] Fetching index...")
    sitemap_list = parse_sitemap(SITEMAP_INDEX)

    print(f"Found {len(sitemap_list)} sitemap files")

    for sm in sitemap_list:
        print("Processing:", sm)

        urls = parse_sitemap(sm)

        for link in urls:
            if link.endswith(".html"):
                all_links.add(link)

        print(f"Collected {len(all_links)} links so far")
        sleep()

    return all_links

if __name__ == "__main__":
    print("Starting CNBC scraper...\n")

    section_links = scrape_sections()
    sitemap_links = scrape_sitemaps()

    all_links = section_links.union(sitemap_links)

    print(f"\nTotal unique links: {len(all_links)}")

    with open("cnbc_links.txt", "w", encoding="utf-8") as f:
        for link in sorted(all_links):
            f.write(link + "\n")

    print("Saved to cnbc_links.txt")

Starting CNBC scraper...


[SECTION] world
Page 1 → 77 links
Page 2 → 77 links
Page 3 → 77 links
Page 4 → 77 links
Page 5 → 77 links
Page 6 → 77 links
Page 7 → 77 links
Page 8 → 77 links
Page 9 → 77 links
Page 10 → 77 links
Page 11 → 77 links
Page 12 → 77 links
Page 13 → 77 links
Page 14 → 77 links
Page 15 → 77 links
Page 16 → 77 links
Page 17 → 77 links
Page 18 → 77 links
Page 19 → 77 links
Page 20 → 77 links

[SECTION] business
Page 1 → 109 links
Page 2 → 143 links
Page 3 → 176 links
Page 4 → 210 links
Page 5 → 244 links
Page 6 → 273 links
Page 7 → 273 links
Page 8 → 273 links
Page 9 → 273 links
Page 10 → 273 links
Page 11 → 273 links
Page 12 → 273 links
Page 13 → 273 links
Page 14 → 273 links
Page 15 → 273 links
Page 16 → 273 links
Page 17 → 273 links
Page 18 → 273 links
Page 19 → 273 links
Page 20 → 273 links

[SECTION] markets
Page 1 → 312 links
Page 2 → 312 links
Page 3 → 312 links
Page 4 → 312 links
Page 5 → 312 links
Page 6 → 312 links
Page 7 → 312 links
Page 8 → 312 links
Page

In [ ]:
import random

file_path = "/content/cnbc_links.txt"

with open(file_path, "r") as f:
    urls = [line.strip() for line in f.readlines() if line.strip()]

print("Total URLs:", len(urls))

sections = {
    "world": [],
    "business": [],
    "markets": [],
    "technology": [],
    "politics": [],
    "economy": []
}

# Categorize by URL text
for url in urls:
    lower = url.lower()

    for sec in sections:
        if sec in lower:
            sections[sec].append(url)

sample_size = 100
final_urls = []

for sec in sections:
    links = list(set(sections[sec]))   # remove duplicates

    if len(links) > sample_size:
        selected = random.sample(links, sample_size)
    else:
        selected = links

    final_urls.extend(selected)

    print(sec.upper(), "→", len(selected))

random.shuffle(final_urls)

# Save txt
with open("/content/cnbc_balanced_links1.txt", "w") as f:
    for link in final_urls:
        f.write(link + "\n")

print("Final:", len(final_urls))

Total URLs: 607452
WORLD → 100
BUSINESS → 100
MARKETS → 100
TECHNOLOGY → 100
POLITICS → 100
ECONOMY → 100
Final: 600


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
from datetime import datetime

#  Step 1: Load URLs
file_path = "/content/cnbc_balanced_links1.txt"

with open(file_path, "r") as f:
    urls = [line.strip() for line in f.readlines() if line.strip()]

print(f"Total URLs loaded: {len(urls)}")

# Step 2: Setup
all_articles = []

headers = {
    "User-Agent": "Mozilla/5.0"
}

#  Step 3: Loop
for idx, url in enumerate(urls):
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")


        headline = None
        author = None
        date = None
        time_val = None
        image = None
        caption = None
        keywords = []
        content = None
        place = None


        json_ld = soup.find("script", type="application/ld+json")

        if json_ld:
            try:
                data = json.loads(json_ld.string)

                if isinstance(data, dict):

                    # Headline
                    headline = data.get("headline")

                    # Author
                    if isinstance(data.get("author"), dict):
                        author = data["author"].get("name")

                    elif isinstance(data.get("author"), list):
                        if len(data["author"]) > 0:
                            author = data["author"][0].get("name")

                    # Date + Time Split
                    raw_datetime = data.get("datePublished")

                    if raw_datetime:
                        try:
                            dt = datetime.fromisoformat(
                                raw_datetime.replace("Z", "+00:00")
                            )
                            date = dt.strftime("%Y-%m-%d")
                            time_val = dt.strftime("%H:%M:%S")
                        except:
                            parts = raw_datetime.split("T")
                            date = parts[0]

                            if len(parts) > 1:
                                time_val = parts[1][:8]

                    # Image
                    image = data.get("image")

                    # Keywords
                    if isinstance(data.get("keywords"), list):
                        keywords = data.get("keywords")

                    elif isinstance(data.get("keywords"), str):
                        keywords = [
                            x.strip()
                            for x in data.get("keywords").split(",")
                        ]

            except:
                pass


        # Headline
        if not headline:
            h1 = soup.find("h1")
            headline = h1.get_text(strip=True) if h1 else None

        # Author
        if not author:
            a = soup.find("a", attrs={"rel": "author"})
            author = a.get_text(strip=True) if a else None

        # Content
        content_div = soup.find("div", {"class": "ArticleBody-articleBody"})

        if content_div:
            content = " ".join(
                [p.get_text(strip=True) for p in content_div.find_all("p")]
            )
        else:
            content = " ".join(
                [p.get_text(strip=True) for p in soup.find_all("p")]
            )

        #  Image + Caption
        figure = soup.find("figure")

        if figure:
            img_tag = figure.find("img")

            if img_tag and img_tag.has_attr("src") and not image:
                image = img_tag["src"]

            cap = figure.find("figcaption")

            if cap:
                caption = cap.get_text(strip=True)


        if not keywords:
            meta_keywords = soup.find("meta", {"name": "keywords"})

            if meta_keywords:
                keywords = [
                    x.strip()
                    for x in meta_keywords.get("content", "").split(",")
                ]

        related_links = set()

        for a in soup.find_all("a", href=True):
            href = a["href"]

            if "cnbc.com" in href and "/202" in href:
                related_links.add(href)

        related_links = list(related_links)[:10]


        meta_place = soup.find("meta", {"name": "location"})

        if meta_place:
            place = meta_place.get("content")

        article_data = {
            "url": url,
            "headline": headline,
            "reporter": author,
            "date": date,
            "time": time_val,
            "content": content,
            "image": image,
            "image_caption": caption,
            "keywords": keywords,
            "related_articles": related_links,
            "place": place
        }

        all_articles.append(article_data)

        print(f"Done {idx+1}/{len(urls)}")

        time.sleep(1)

    except Exception as e:
        print(f"Error at {url}: {e}")

# - Step 4: Save JSON -
output_file = "/content/cnbc_articles.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(all_articles, f, indent=4, ensure_ascii=False)

print(f"\nData saved successfully to {output_file}")

Total URLs loaded: 600
Done 1/600
Done 2/600
Done 3/600
Done 4/600
Done 5/600
Done 6/600
Done 7/600
Done 8/600
Done 9/600
Done 10/600
Done 11/600
Done 12/600
Done 13/600
Done 14/600
Done 15/600
Done 16/600
Done 17/600
Done 18/600
Done 19/600
Done 20/600
Done 21/600
Done 22/600
Done 23/600
Done 24/600
Done 25/600
Done 26/600
Done 27/600
Done 28/600
Done 29/600
Done 30/600
Done 31/600
Done 32/600
Done 33/600
Done 34/600
Done 35/600
Done 36/600
Done 37/600
Done 38/600
Done 39/600
Done 40/600
Done 41/600
Done 42/600
Done 43/600
Done 44/600
Done 45/600
Done 46/600
Done 47/600
Done 48/600
Done 49/600
Done 50/600
Done 51/600
Done 52/600
Done 53/600
Done 54/600
Done 55/600
Done 56/600
Done 57/600
Done 58/600
Done 59/600
Done 60/600
Done 61/600
Done 62/600
Done 63/600
Done 64/600
Done 65/600
Done 66/600
Done 67/600
Done 68/600
Done 69/600
Done 70/600
Done 71/600
Done 72/600
Error at https://www.cnbc.com/2020/03/11/bank-of-americas-top-technology-officer-howard-boville-has-left-source-says.html: